In [4]:
import tiktoken
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

from cs336_basics import train_model, checkpoint

import importlib
importlib.reload(checkpoint)
importlib.reload(train_model)



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [5]:
from itertools import chain
from datasets import load_dataset, Features, List, Value

tokenizer = tiktoken.get_encoding("gpt2")
vocab_size = tokenizer.n_vocab
context_length = 128

# Load lines into an Arrow cache; original line breaks are not preserved.
dataset = load_dataset(
    "text",
    data_files="data/TinyStoriesV2-GPT4-train.txt",
    split="train",
    cache_dir="data/hf_cache",
)

def tokenize(batch):
    return {
        "input_ids": [
            tokenizer.encode(text, allowed_special={"<|endoftext|>"})
            for text in batch["text"]
        ]
    }

def group_tokens(batch):
    ids = list(chain.from_iterable(batch["input_ids"]))
    # Drop the incomplete tail of each preprocessing batch.
    return {
        "input_ids": [
            ids[i:i + context_length + 1]
            for i in range(0, len(ids) - context_length, context_length)
        ]
    }

# Repeated runs reuse these cached transformations.
tokenized = dataset.map(
    tokenize, batched=True, remove_columns=["text"],
)
train_dataset = tokenized.map(
    group_tokens,
    batched=True,
    features=Features({
        "input_ids": List(Value("uint16"), length=context_length + 1),
    }),
)
print(train_dataset)
print(f"training tokens per epoch: {len(train_dataset) * context_length:,}")


Map: 100%|██████████| 15600057/15600057 [00:51<00:00, 300612.95 examples/s]

Dataset({
    features: ['input_ids'],
    num_rows: 4151413
})
training tokens per epoch: 531,380,864


In [6]:
# For a quick check, use train_dataset.select(range(min(100, len(train_dataset)))).
model, optimizer, checkpoint = train_model.train(
    dataset=train_dataset, vocab_size=vocab_size, num_epoch=3
)


tokens_per_epoch=531,380,864 samples=4,151,413 steps_per_epoch=518,927 total_steps=1,556,781
epoch=1 step=20 loss=10.3381
epoch=1 step=40 loss=8.9218
epoch=1 step=60 loss=7.7509
epoch=1 step=80 loss=6.8784
epoch=1 step=100 loss=6.3209
epoch=1 step=120 loss=6.1187
epoch=1 step=140 loss=6.0550
epoch=1 step=160 loss=5.9455
epoch=1 step=180 loss=5.9034
epoch=1 step=200 loss=5.7955
epoch=1 step=220 loss=5.8648
epoch=1 step=240 loss=5.4824
epoch=1 step=260 loss=5.5728
epoch=1 step=280 loss=5.5402
epoch=1 step=300 loss=5.1856
epoch=1 step=320 loss=5.0686
epoch=1 step=340 loss=4.9373
epoch=1 step=360 loss=4.7762
epoch=1 step=380 loss=5.0885
epoch=1 step=400 loss=5.2068
epoch=1 step=420 loss=5.1822
epoch=1 step=440 loss=4.5735
epoch=1 step=460 loss=4.7595
epoch=1 step=480 loss=4.5193
epoch=1 step=500 loss=4.6799
epoch=1 step=520 loss=4.6696
epoch=1 step=540 loss=4.3493
epoch=1 step=560 loss=4.4088
epoch=1 step=580 loss=4.7642
epoch=1 step=600 loss=4.2048
epoch=1 step=620 loss=4.3552
epoch=1 ste

KeyboardInterrupt: 